inspect the table  silver customers 


In [0]:
%python
from pyspark.sql.functions import col , sum

In [0]:
%python
df = spark.table("ecommerce.bronze.orders")

df.show(10)


In [0]:
%python
print("Total rows:", df.count())
print("Distinct order_id:", df.select("order_id").distinct().count())

----------------------    1 row = 1 order    ----------------------    

In [0]:
%python
df.groupBy("order_status") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

- +------------+-----+
- |order_status|count|
- +------------+-----+
- |   delivered|96512|
- |     shipped| 1078|
- |    canceled|  625|
- | unavailable|  609|
- |  processing|  350|
- |    invoiced|  314|
- |    approved|   65|
- |     created|   58|
- +------------+-----+
- 

In [0]:
%python
for c in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]:
    print(c, df.filter(col(c).isNull()).count())

- > order_purchase_timestamp 0
- > order_approved_at 214
- > order_delivered_carrier_date 1953
- > order_delivered_customer_date 3135
- > order_estimated_delivery_date 0
- 

In [0]:
%python


df.filter(
    (col("order_status") == "delivered") &
    col("order_delivered_customer_date").isNull()
).count()

In [0]:
%python
df.filter(
    (col("order_status") == "delivered") &
    col("order_delivered_carrier_date").isNull()
).count()

In [0]:
%python
customer_ids = df.select("customer_id").distinct()

customer_ids.join(
    spark.table("ecommerce.bronze.customers")
    .select("customer_id")
    .distinct(),
    "customer_id",
    "left_anti"
).count()

data-quality check

In [0]:
%python


df.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
).count()

In [0]:
%python
df.filter(
    col("order_delivered_carrier_date") < col("order_purchase_timestamp")
).count()

In [0]:
%python
df.filter(
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
).count()

In [0]:
%python
df.filter(
    col("order_delivered_customer_date") < col("order_delivered_carrier_date")
).count()

In [0]:
%python
df.filter(
    col("order_delivered_carrier_date") < col("order_purchase_timestamp")
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)


In [0]:
%python
df.filter(
    col("order_delivered_customer_date") < col("order_delivered_carrier_date")
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)